# Profile likelihood ratio confidence bounds and MSE

I'm minimising the mean squared error

\begin{align}
E(\theta) = \frac{1}{n}\sum(v_i - f(\theta)_i)^2
\end{align}

and want to calculate confidence intervals.

For fitting multiple exponentials, I've read it's a good idea to use a profile likelihood ratio method, which means reformulating as a statistical problem and assuming some noise model.

Even though it's definitely not right for our data, I've written it out as an additive normally distributed variable, for:
\begin{align}
L(\theta, \sigma | v) \equiv p(v | \theta, \sigma)
 &= \prod^n_{i=1} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( \frac{(v_i - f(\theta)_i)^2}{2\sigma^2} \right)
\end{align}
for
\begin{align}
\mathcal{l}(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v)
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum^n_{i=1} (v_i - f(\theta)_i)^2
\end{align}

Having found the optimum $\theta_\text{best}$, we want to test other values $\theta_\text{test}$ by fixing a parameter of interest, re-running the optimisation, and then comparing with a [likelihood ratio test](https://en.wikipedia.org/wiki/Likelihood-ratio_test):

\begin{align}
-2 \log\left(\frac{L_\text{test}}{L_\text{best}}\right) < \chi^2_{1,1-\alpha}
\end{align}

where $\chi_{1,1-\alpha}$ is the value [Chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) with $1$ degree of freedom and $1-\alpha$ is the level of confidence (i.e. for 5% we get $\chi_{1,0.95}$).

### Rewrite in terms of $E$

We can rewrite as log likelihoods

\begin{align}
-2 \log\left(\frac{L_\text{test}}{L_\text{best}}\right) = -2 (\mathcal{l}_\text{test} - \mathcal{l}_\text{best})
\end{align}

and then, because $n$ and $\sigma$ are unchanged, we get
\begin{align}
-2 \cdot \frac{-1}{2\sigma^2} \left(\sum (v_i - f(\theta_\text{test})_i)^2 - \sum (v_i - f(\theta_\text{best})_i)^2\right)
= \frac{n}{\sigma^2} (E_\text{test} - E_\text{best})
\end{align}

To approximate $\sigma^2$ (I'm not inferring it), we take the (uncorrected, should be ok because $n$ is large) sample standard deviation of the residuals, which is just $E_\text{best}$:

\begin{align}
\frac{n}{\sigma^2} (E_\text{test} - E_\text{best}) \approx n \left(\frac{E_\text{test}}{E_\text{best}} - 1\right) < \chi^2_{1,1-\alpha}
\end{align}

which would mean our confidence interval includes all values for which

\begin{align}
E_\text{test} < (1 + \chi^2_{1,1-\alpha} / n) E_\text{best}
\end{align}

The appearance of $n$ is a bit jarring, but it makes sense: more data means tighter bounds.

### Getting the test statistic

We can look it up in a table, or ask Scipy:

In [1]:
import scipy
print(scipy.stats.chi2.ppf(0.99, 1))
print(scipy.stats.chi2.ppf(0.95, 1))
print(scipy.stats.chi2.ppf(0.90, 1))

6.6348966010212145
3.841458820694124
2.705543454095404


In [2]:
print(scipy.stats.chi2.ppf(0.01, 1))
print(scipy.stats.chi2.ppf(0.05, 1))
print(scipy.stats.chi2.ppf(0.10, 1))

0.00015708785790970184
0.003932140000019522
0.01579077409343122


**Everything I read suggests I should be using e.g. 0.95 for 5% intervals. But this doesn't make sense: a larger $\chi$ gives me a larger region**